# Библиотека Transformers. Детальный разбор того, как работают трансформеры.

In [1]:
from transformers import AutoTokenizer
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_inputs = [
    "Hello, my name is Slava. I am from Russia.",
    "My life is cool",
]
model = AutoModel.from_pretrained(checkpoint)

inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[  101,  7592,  1010,  2026,  2171,  2003, 22889, 12462,  1012,  1045,
          2572,  2013,  3607,  1012,   102],
        [  101,  2026,  2166,  2003,  4658,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}
torch.Size([2, 15, 768])


In [2]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import torch


raw_input = ["Hello, my name is Slava. I am from Russia.",
    "My life is cool",]

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
outputs = model(**inputs)


print(outputs.logits.shape)
print(outputs.logits)

prediction = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(prediction)

markers = model.config.id2label
print(markers)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

torch.Size([2, 2])
tensor([[-2.7252,  2.8447],
        [-3.9835,  4.3535]], grad_fn=<AddmmBackward0>)
tensor([[3.7963e-03, 9.9620e-01],
        [2.3942e-04, 9.9976e-01]], grad_fn=<SoftmaxBackward0>)
{0: 'NEGATIVE', 1: 'POSITIVE'}


In [3]:
from transformers import BertConfig, BertModel

config = BertConfig()

model = BertModel(config)

print(config)

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.19.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [4]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased")

model.save_pretrained("/Users/xebecedex/xbdx/models/bert")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
import torch

sequences = ["Hello", "Nice", "Cool"]
encoded_sequences = [
    [101, 7592, 999, 102],
    [101, 4658, 1012, 102],
    [101, 3835, 999, 102],
]

model_inputs = torch.tensor(encoded_sequences)

output = model(model_inputs)

print(output)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 4.4496e-01,  4.8276e-01,  2.7797e-01,  ..., -5.4032e-02,
           3.9394e-01, -9.4770e-02],
         [ 2.4943e-01, -4.4093e-01,  8.1772e-01,  ..., -3.1917e-01,
           2.2992e-01, -4.1172e-02],
         [ 1.3668e-01,  2.2518e-01,  1.4502e-01,  ..., -4.6915e-02,
           2.8224e-01,  7.5566e-02],
         [ 1.1789e+00,  1.6738e-01, -1.8187e-01,  ...,  2.4671e-01,
           1.0441e+00, -6.1976e-03]],

        [[ 3.6436e-01,  3.2464e-02,  2.0258e-01,  ...,  6.0111e-02,
           3.2451e-01, -2.0996e-02],
         [ 7.1866e-01, -4.8725e-01,  5.1740e-01,  ..., -4.4012e-01,
           1.4553e-01, -3.7545e-02],
         [ 3.3223e-01, -2.3271e-01,  9.4876e-02,  ..., -2.5268e-01,
           3.2172e-01,  8.1095e-04],
         [ 1.2523e+00,  3.5754e-01, -5.1320e-02,  ..., -3.7840e-01,
           1.0526e+00, -5.6255e-01]],

        [[ 2.4042e-01,  1.4718e-01,  1.2110e-01,  ...,  7.6062e-02,
           3.3564e-01,  2

## Токенизатор


In [6]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
sequence = "Использование onescript может помочь автоматизировать DevOps 1С"

tokens = tokenizer.tokenize(sequence)
tokenizer(sequence)

print(tokens)

['И', '##с', '##п', '##о', '##л', '##ь', '##з', '##ов', '##а', '##н', '##и', '##е', 'ones', '##cript', 'м', '##о', '##ж', '##е', '##т', 'п', '##о', '##м', '##о', '##ч', '##ь', 'а', '##в', '##т', '##о', '##м', '##а', '##т', '##и', '##з', '##и', '##р', '##ов', '##а', '##т', '##ь', 'Dev', '##O', '##ps', '1', '##С']


In [7]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[456, 28403, 28402, 16948, 28400, 28414, 28398, 24625, 10286, 17127, 17424, 19692, 3200, 13590, 487, 16948, 28397, 19692, 28404, 490, 16948, 28401, 16948, 28409, 28414, 475, 28394, 28404, 16948, 28401, 10286, 28404, 17424, 28398, 17424, 20442, 24625, 10286, 28404, 28414, 18433, 2346, 3491, 122, 28382]


In [8]:
decoded_string = tokenizer.decode(ids)

print(decoded_string)

Использование onescript может помочь автоматизировать DevOps 1С


В предыдущем листинге мы увидели, как последовательность слов преобразуется в список чисел. Теперь преобразуем список чисел в тензор и передадим его в модель

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "Использование onescript может помочь автоматизировать DevOps 1С"

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)

tokenized_input = tokenizer(sequence, return_tensors="pt")
print(tokenized_input["input_ids"])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[  101,  1188, 29747, 29746, 14150, 29436, 23742, 29744, 19259, 28995,
         10325, 15290,  3924, 23235,  1191, 14150, 29743, 15290, 22919,  1194,
         14150, 29745, 14150, 29752, 23742,  1180, 25529, 22919, 14150, 29745,
         10260, 22919, 10325, 29744, 10325, 16856, 19259, 10260, 22919, 23742,
         16475, 11923,  1015, 29747,   102]])


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "Использование onescript может помочь автоматизировать DevOps 1С"

# 1. Одиночный текст (уже со специальными токенами CLS/SEP)
tokenized_input = tokenizer(sequence, return_tensors="pt")
print("Ключи в tokenized_input:", tokenized_input.keys()) # dict_keys(['input_ids', 'attention_mask'])

output = model(**tokenized_input)
print("Logits:", output.logits)


# 2. Правильная пакетная обработка (Batching) нескольких строк
# Для батчей всегда нужно передавать список строк и включать padding=True
batched_sequences = [
    "Использование onescript может помочь автоматизировать DevOps 1С",
    "Второе предложение для проверки батча"
]

# Токенизатор сам сделает паддинг и вернет правильные тензоры
batched_input = tokenizer(batched_sequences, padding=True, return_tensors="pt")

# Теперь можно безопасно передавать в модель
batched_output = model(**batched_input)
print("Batched Logits:", batched_output.logits)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Ключи в tokenized_input: KeysView({'input_ids': tensor([[  101,  1188, 29747, 29746, 14150, 29436, 23742, 29744, 19259, 28995,
         10325, 15290,  3924, 23235,  1191, 14150, 29743, 15290, 22919,  1194,
         14150, 29745, 14150, 29752, 23742,  1180, 25529, 22919, 14150, 29745,
         10260, 22919, 10325, 29744, 10325, 16856, 19259, 10260, 22919, 23742,
         16475, 11923,  1015, 29747,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
Logits: tensor([[ 0.4775, -0.2215]], grad_fn=<AddmmBackward0>)
Batched Logits: tensor([[ 0.4775, -0.2215],
        [-0.3437,  0.5679]], grad_fn=<AddmmBackward0>)


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]] 

batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]] 

batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

#print(model(torch.tensor(sequence1_ids)).logits)
#print(model(torch.tensor(sequence2_ids)).logits)
#print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
